In [ ]:
import pandas as pd
from pathlib import Path
import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

from se_coast_strandings import (
    make_dt_col,
    make_cyclic,
    make_season_col,
    make_cyclic_season,
)

In [ ]:
data_dir = Path("../data/raw")

In [ ]:
df = pd.read_excel(data_dir / "UNC-DataRequest-01302026.xlsx", sheet_name="2015-2024")

In [ ]:
print(df.columns)

In [ ]:
df["Latitude Actual/Estimate"].value_counts(dropna=False)

In [ ]:
same = df["Latitude Actual/Estimate"] == df["Longitude Actual/Estimate"]
df = df[same]

In [ ]:
df["Longitude"] = pd.to_numeric(df["Longitude"], errors="coerce")
df["Latitude"] = pd.to_numeric(df["Latitude"], errors="coerce")
df = df[~df["Longitude"].isna() & ~df["Latitude"].isna()]

In [ ]:
df["is_actual_coords"] = df["Latitude Actual/Estimate"] == "actual"

In [ ]:
df["is_actual_coords"].value_counts(dropna=False)

In [ ]:
df[["Day of Observation", "Month of Observation", "Year of Observation"]].head(5)

In [ ]:
df["mms_observation_dt"] = make_dt_col(
    df["Day of Observation"], df["Month of Observation"], df["Year of Observation"]
)

In [ ]:
df["dayofweek_sin"], df["dayofweek_cos"] = make_cyclic(
    df["mms_observation_dt"].dt.dayofweek, 7, name="dayofweek"
)
df["month_sin"], df["month_cos"] = make_cyclic(
    df["mms_observation_dt"].dt.month, 12, name="month"
)

In [ ]:
df["season"] = make_season_col(df["mms_observation_dt"])
df["season_sin"], df["season_cos"] = make_cyclic_season(
    df["mms_observation_dt"], name="season"
)

In [ ]:
df["dayofyear_sin"], df["dayofyear_cos"] = make_cyclic(
    df["mms_observation_dt"].dt.dayofyear, 365, name="dayofyear"
)

In [ ]:
cyclic_features = df[
    [c for c in df.columns if c.endswith("_sin") or c.endswith("_cos")]
]
cyclic_features

## Coordinate Reference System (CRS) Note

All strandings operations in this notebook use **EPSG:4326** (WGS 84, decimal degrees).
This is the native CRS of the NOAA strandings coordinates. The `assign_region()` function
used in the modeling pipeline also operates on raw decimal-degree latitudes — no reprojection
is needed.

Note: Notebook `01_c_plankton_data_clean` uses **EPSG:5070** (NAD83 / Conus Albers) for
spatial buffering operations only. That CRS choice does not affect the strandings pipeline.

In [ ]:
gdf = gpd.GeoDataFrame(
    df,
    geometry=gpd.points_from_xy(df["Longitude"], df["Latitude"]),
    crs="EPSG:4326",
)

In [ ]:
states = gpd.read_file("../data/reference/cb_2018_us_state_5m.shp").set_crs("EPSG:4326")
states.shape, states.crs

In [ ]:
se_coast_states = states[
    states["NAME"].isin(
        [
            "Virginia",
            "North Carolina",
            "South Carolina",
        ]
    )
]
se_coast_states

In [ ]:
gdf["_t"] = mdates.date2num(gdf["mms_observation_dt"])

In [ ]:
out_path = Path("../figures/jh_strandings_dataset_nc_map.png")

fig, ax = plt.subplots()

se_coast_states.loc[se_coast_states["STUSPS"] == "NC"].plot(
    ax=ax,
    edgecolor="black",
    facecolor="none",
    linewidth=0.5,
    figsize=(12, 12),
)

gdf.plot(
    ax=ax,
    column="_t",
    markersize=3,
    cmap="viridis",
    alpha=0.5,
)

ax.set_xlim(-78.75, -74.25)
ax.set_ylim(33.5, 36.75)
ax.set_title("Stranding locations along the SE Coast from 2015 to 2025", pad=20)
ax.set_xlabel("Longitude (degrees, EPSG:4326)")
ax.set_ylabel("Latitude (degrees, EPSG:4326)")

plt.savefig(out_path)
plt.show()

In [ ]:
out_path = Path("../figures/jh_strandings_dataset_map.png")


ax = states.plot(edgecolor="black", facecolor="none", linewidth=0.5, figsize=(8, 8))
gdf.plot(ax=ax, column="_t", markersize=1.5, cmap="viridis", alpha=0.5)

ax.set_xlim(-82, -74)
ax.set_ylim(32, 38.5)
ax.set_title("Stranding locations along the SE Coast from 2015 to 2025", pad=20)
ax.set_xlabel("Longitude (degrees, EPSG:4326)")
ax.set_ylabel("Latitude (degrees, EPSG:4326)")
plt.savefig(out_path, dpi=300)
